In [1]:
!pip install tensorflow


In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder



In [4]:
# Load dataset generated earlier
df = pd.read_csv('amazon_product_reviews.csv')

# Keep only necessary columns and drop nulls in ReviewText
df = df[['ReviewText', 'SentimentLabel']].dropna()

# Encode labels (Negative: 0, Neutral: 1, Positive: 2)
label_encoder = LabelEncoder()
df['Label'] = label_encoder.fit_transform(df['SentimentLabel'])

# To make training fast for this CA, we will sample 1000 rows. 
# Remove the `.sample()` in production to use the full dataset.
df_sample = df.sample(1000, random_state=42)

X = df_sample['ReviewText'].values
y = df_sample['Label'].values

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 800
Testing samples: 200


In [5]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# Tokenization and Padding for LSTM
vocab_size = 5000
max_len = 64

tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=max_len, padding='post')
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=max_len, padding='post')

# Build LSTM Model
lstm_model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64, input_length=max_len),
    LSTM(64, return_sequences=False),
    Dropout(0.5),
    Dense(3, activation='softmax') # 3 classes: Neg, Neu, Pos
])

lstm_model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train LSTM (Set epochs=5 or more for better results)
print("Training LSTM Baseline...")
lstm_history = lstm_model.fit(X_train_seq, y_train, epochs=3, validation_data=(X_test_seq, y_test), batch_size=32, verbose=1)


c:\Users\krish\python\venv\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Training LSTM Baseline...
Epoch 1/3
25/25 ━━━━━━━━━━━━━━━━━━━━ 7s 87ms/step - accuracy: 0.5412 - loss: 1.0201 - val_accuracy: 0.4650 - val_loss: 1.0602
Epoch 2/3
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - accuracy: 0.5562 - loss: 1.0076 - val_accuracy: 0.4650 - val_loss: 1.0789
Epoch 3/3
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 65ms/step - accuracy: 0.5562 - loss: 0.9989 - val_accuracy: 0.4650 - val_loss: 1.0640


In [6]:
!pip install transformers

In [14]:
!pip install tensorflow
!pip install --upgrade transformers

In [ ]:
!pip install torch accelerate transformers

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# 1. Load DistilBERT tokenizer
model_name = 'distilbert-base-uncased'
bert_tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Tokenize texts
train_encodings = bert_tokenizer(list(X_train), truncation=True, padding=True, max_length=max_len)
test_encodings = bert_tokenizer(list(X_test), truncation=True, padding=True, max_length=max_len)

# 3. Create a simple Dataset class for PyTorch
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = ReviewDataset(train_encodings, y_train)
test_dataset = ReviewDataset(test_encodings, y_test)

# 4. Load Pre-trained BERT Model (PyTorch version)
bert_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# 5. Setup Training Arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch", # Evaluates at the end of the epoch
    logging_dir='./logs',
)

# 6. Initialize Trainer
trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

# 7. Fine-tune BERT
print("\nFine-tuning BERT...")
trainer.train()

ImportError: cannot import name 'TFAutoModelForSequenceClassification' from 'transformers' (c:\Users\krish\python\venv\Lib\site-packages\transformers\__init__.py)